# 03_silver_era5.ipynb — ERA5 precipitation-only + ERA5_PROXY

Este notebook procesa los NetCDF mensuales de **ERA5 (Copernicus CDS)** para generar:

- `silver/ocean_hourly/source=ERA5/...`
- `silver/meteo_hourly/source=ERA5/...`

Requiere que ya existan:

```text
silver/beach_geography/beach_geography.parquet
silver/ocean_hourly/source=SIMAR/...
silver/meteo_hourly/source=SIMAR/...
silver/ocean_physics/source=SIMAR/...
```

La tabla `beach_geography` se usa como `dim_zone` para asignar cada punto de malla ERA5 a la zona costera/playa más cercana.

Variables esperadas en ERA5:

```text
u10, v10, sp, t2m, tp, swh, mwp, mwd
```

El notebook es incremental: procesa archivo a archivo para evitar cargar los 25 años completos en memoria.

## Celda 0 — Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Celda 1 — Instalar librerías necesarias

In [ ]:
!pip -q install xarray netCDF4 h5netcdf scipy dask geopandas pyarrow shapely fiona tqdm cfgrib eccodes

## Celda 2 — Imports, rutas y configuración

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd
import xarray as xr
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import re
import unicodedata
import json
import shutil
import gc
from tqdm.auto import tqdm

BASE_DIR = Path("/content/drive/MyDrive/AI Projects/DeepWave Canarias")
BRONZE_DIR = BASE_DIR / "data/bronze"
SILVER_DIR = BASE_DIR / "silver"

ERA5_DIR = BRONZE_DIR / "ERA5 (Copernicus CDS)"
DIM_ZONE_PATH = SILVER_DIR / "beach_geography" / "beach_geography.parquet"

OUT_OCEAN_DIR = SILVER_DIR / "ocean_hourly"
OUT_METEO_DIR = SILVER_DIR / "meteo_hourly"

QC_DIR = SILVER_DIR / "_quality_reports"
META_DIR = SILVER_DIR / "_metadata"

QC_DIR.mkdir(parents=True, exist_ok=True)
META_DIR.mkdir(parents=True, exist_ok=True)
OUT_OCEAN_DIR.mkdir(parents=True, exist_ok=True)
OUT_METEO_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_NAME = "ERA5"

BBOX_CANARIAS = {
    "lat_min": 27.0,
    "lat_max": 29.5,
    "lon_min": -18.5,
    "lon_max": -13.0,
}

MIN_VALID_TS = pd.Timestamp("1999-01-01", tz="UTC")
MAX_VALID_TS = pd.Timestamp("2031-01-01", tz="UTC")

# Cambia esto para pruebas rápidas. Usa None para procesar todo.
MAX_FILES_FOR_TEST = None

print("BASE_DIR:", BASE_DIR)
print("ERA5_DIR existe:", ERA5_DIR.exists())
print("DIM_ZONE_PATH existe:", DIM_ZONE_PATH.exists())

if not ERA5_DIR.exists():
    raise FileNotFoundError(f"No existe ERA5_DIR: {ERA5_DIR}")

if not DIM_ZONE_PATH.exists():
    raise FileNotFoundError("No existe beach_geography.parquet. Ejecuta primero 01_silver_dim_zone.ipynb.")

BASE_DIR: /content/drive/MyDrive/AI Projects/DeepWave Canarias
ERA5_DIR existe: True
DIM_ZONE_PATH existe: True


## Celda 3 — Utilidades generales

In [ ]:
def normalize_text(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip()
    value = unicodedata.normalize("NFKD", value)
    value = "".join(c for c in value if not unicodedata.combining(c))
    value = re.sub(r"\s+", " ", value)
    return value.upper()


def normalize_col(col):
    col = normalize_text(col)
    if pd.isna(col):
        return ""
    col = re.sub(r"[^A-Z0-9]+", "_", col)
    col = re.sub(r"_+", "_", col).strip("_")
    return col


def slugify(value):
    value = normalize_text(value)
    if pd.isna(value):
        return "UNKNOWN"
    value = re.sub(r"[^A-Z0-9]+", "_", value)
    value = re.sub(r"_+", "_", value)
    return value.strip("_")


def standardize_longitudes_to_180(lon_values):
    """
    Convierte longitudes 0..360 a -180..180.
    """
    lon = np.asarray(lon_values)
    return ((lon + 180) % 360) - 180


def make_era5_point_id(lat, lon):
    return "ERA5_" + pd.Series(lat).round(4).astype(str) + "_" + pd.Series(lon).round(4).astype(str)


def wind_speed_from_uv(u, v):
    return np.sqrt(u ** 2 + v ** 2)


def wind_direction_from_uv(u, v):
    """
    Dirección meteorológica desde donde sopla el viento, grados desde el norte.
    u/v son componentes hacia el este/norte.
    """
    return (np.degrees(np.arctan2(-u, -v)) + 360) % 360


def ensure_utc(series):
    return pd.to_datetime(series, utc=True, errors="coerce")


def safe_float_stats(series):
    s = pd.to_numeric(series, errors="coerce")
    return {
        "min": float(s.min()) if s.notna().any() else np.nan,
        "max": float(s.max()) if s.notna().any() else np.nan,
        "mean": float(s.mean()) if s.notna().any() else np.nan,
        "missing_pct": float(s.isna().mean() * 100) if len(s) else 0.0,
    }

## Celda 4 — Cargar `dim_zone` / `beach_geography`

In [ ]:
beach_geography = pd.read_parquet(DIM_ZONE_PATH)

required_zone_cols = ["zona_id", "nombre_zona", "isla", "municipio", "lat", "lon"]
missing_zone_cols = [c for c in required_zone_cols if c not in beach_geography.columns]

if missing_zone_cols:
    raise ValueError(f"Faltan columnas en beach_geography: {missing_zone_cols}")

print("beach_geography shape:", beach_geography.shape)
print("zona_id únicos:", beach_geography["zona_id"].nunique())

display(beach_geography.head())

gdf_zones = gpd.GeoDataFrame(
    beach_geography.copy(),
    geometry=gpd.points_from_xy(beach_geography["lon"], beach_geography["lat"]),
    crs="EPSG:4326",
)

gdf_zones_m = gdf_zones.to_crs("EPSG:3857")

beach_geography shape: (561, 17)
zona_id únicos: 561


,zona_id,nombre_zona,isla,municipio,lat,lon,tipo_zona,orientacion_costa,exposicion_norte,exposicion_oeste,exposicion_este,exposicion_swell_nw,exposicion_swell_ne,vulnerabilidad_costera,vulnerabilidad_source,spatial_match_isla,spatial_match_municipio
0,CAN_TF_EL_PUERTITO_0,El Puertito,Tenerife,Güímar,28.2923,-16.3766,playa,NW,1,1,0,1,1,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True
1,CAN_EH_LA_RESTINGA,La Restinga,El Hierro,El Pinar de El Hierro,27.6408,-17.9799,playa,W,0,1,0,1,0,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True
2,CAN_EH_ARENAS_BLANCAS,Arenas Blancas,El Hierro,Frontera,27.7667,-18.1218,playa,W,0,1,0,1,0,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True
3,CAN_EH_EL_VERODAL,El Verodal,El Hierro,Frontera,27.7471,-18.1512,playa,W,0,1,0,1,0,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True
4,CAN_EH_CHARCO_AZUL_0,Charco Azul,El Hierro,Frontera,27.7563,-18.0990,playa,W,0,1,0,1,0,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True


## Celda 5 — Localizar archivos ERA5

In [ ]:
era5_files = sorted(ERA5_DIR.rglob("era5_canarias_*.nc"))

if MAX_FILES_FOR_TEST is not None:
    era5_files = era5_files[:MAX_FILES_FOR_TEST]

if not era5_files:
    raise FileNotFoundError(f"No se encontraron NetCDF ERA5 en {ERA5_DIR}")

ERA5_FILENAME_RE = re.compile(r"era5_canarias_(?P<year>\d{4})_(?P<month>\d{2})\.nc$")

rows = []

for p in era5_files:
    m = ERA5_FILENAME_RE.search(p.name)
    rows.append(
        {
            "path": str(p),
            "filename": p.name,
            "year_from_path": int(m.group("year")) if m else None,
            "month_from_path": int(m.group("month")) if m else None,
            "size_mb": round(p.stat().st_size / 1024 / 1024, 2),
        }
    )

era5_files_df = pd.DataFrame(rows)

print("Archivos ERA5 encontrados:", len(era5_files_df))
display(era5_files_df.head(20))
display(era5_files_df.tail(5))

print("Años encontrados:")
display(era5_files_df["year_from_path"].value_counts().sort_index().reset_index())

Archivos ERA5 encontrados: 300


,path,filename,year_from_path,month_from_path,size_mb
0,/content/drive/MyDrive/AI Projects/DeepWave Ca...,era5_canarias_2001_01.nc,2001,1,3.03
1,/content/drive/MyDrive/AI Projects/DeepWave Ca...,era5_canarias_2001_02.nc,2001,2,2.73
2,/content/drive/MyDrive/AI Projects/DeepWave Ca...,era5_canarias_2001_03.nc,2001,3,3.05
3,/content/drive/MyDrive/AI Projects/DeepWave Ca...,era5_canarias_2001_04.nc,2001,4,2.97
4,/content/drive/MyDrive/AI Projects/DeepWave Ca...,era5_canarias_2001_05.nc,2001,5,3.08
5,/content/drive/MyDrive/AI Projects/DeepWave Ca...,era5_canarias_2001_06.nc,2001,6,2.97
6,/content/drive/MyDrive/AI Projects/DeepWave Ca...,era5_canarias_2001_07.nc,2001,7,3.09
7,/content/drive/MyDrive/AI Projects/DeepWave Ca...,era5_canarias_2001_08.nc,2001,8,3.07
8,/content/drive/MyDrive/AI Projects/DeepWave Ca...,era5_canarias_2001_09.nc,2001,9,2.99
9,/content/drive/MyDrive/AI Projects/DeepWave Ca...,era5_canarias_2001_10.nc,2001,10,3.07


,path,filename,year_from_path,month_from_path,size_mb
295,/content/drive/MyDrive/AI Projects/DeepWave Ca...,era5_canarias_2025_08.nc,2025,8,3.09
296,/content/drive/MyDrive/AI Projects/DeepWave Ca...,era5_canarias_2025_09.nc,2025,9,3.01
297,/content/drive/MyDrive/AI Projects/DeepWave Ca...,era5_canarias_2025_10.nc,2025,10,3.12
298,/content/drive/MyDrive/AI Projects/DeepWave Ca...,era5_canarias_2025_11.nc,2025,11,2.98
299,/content/drive/MyDrive/AI Projects/DeepWave Ca...,era5_canarias_2025_12.nc,2025,12,3.10


Años encontrados:


,year_from_path,count
0,2001,12
1,2002,12
2,2003,12
3,2004,12
4,2005,12
5,2006,12
6,2007,12
7,2008,12
8,2009,12
9,2010,12


## Celda 6 — Inspección rápida de variables NetCDF

In [ ]:
inspect_rows = []

for p in era5_files[:min(3, len(era5_files))]:
    try:
        with xr.open_dataset(p) as ds0:
            inspect_rows.append(
                {
                    "filename": p.name,
                    "dims": dict(ds0.dims),
                    "coords": list(ds0.coords),
                    "data_vars": list(ds0.data_vars),
                }
            )
    except Exception as e:
        inspect_rows.append(
            {
                "filename": p.name,
                "error": repr(e),
            }
        )

inspect_df = pd.DataFrame(inspect_rows)
display(inspect_df)

,filename,error
0,era5_canarias_2001_01.nc,"ValueError(""did not find a match in any of xar..."
1,era5_canarias_2001_02.nc,"ValueError(""did not find a match in any of xar..."
2,era5_canarias_2001_03.nc,"ValueError(""did not find a match in any of xar..."


## Celda 7 — Funciones robustas para abrir, recortar y convertir ERA5

In [ ]:
ERA5_VAR_ALIASES = {
    # viento y meteo
    "u10": ["u10", "10u", "10m_u_component_of_wind", "u_component_of_wind_10m"],
    "v10": ["v10", "10v", "10m_v_component_of_wind", "v_component_of_wind_10m"],
    "sp": ["sp", "surface_pressure", "msl"],
    "t2m": ["t2m", "2t", "2m_temperature", "temperature_2m"],
    "tp": ["tp", "total_precipitation"],

    # oleaje
    "swh": [
        "swh",
        "significant_height_of_combined_wind_waves_and_swell",
        "significant_wave_height",
    ],
    "mwp": [
        "mwp",
        "mean_wave_period",
        "mean_wave_period_based_on_first_moment",
    ],
    "mwd": [
        "mwd",
        "mean_wave_direction",
    ],
}


TMP_ERA5_EXTRACT_DIR = Path("/content/era5_tmp_extract")
TMP_ERA5_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)


def get_file_magic(path, n=16):
    with open(path, "rb") as f:
        return f.read(n)


def open_era5_any(path):
    """
    Abre ERA5 aunque el archivo .nc sea realmente NetCDF, GRIB o ZIP.
    """

    path = Path(path)
    magic = get_file_magic(path)

    # Caso ZIP
    if magic.startswith(b"PK"):
        import zipfile

        extract_dir = TMP_ERA5_EXTRACT_DIR / path.stem
        extract_dir.mkdir(parents=True, exist_ok=True)

        with zipfile.ZipFile(path, "r") as z:
            z.extractall(extract_dir)

        candidates = (
            list(extract_dir.rglob("*.nc"))
            + list(extract_dir.rglob("*.grib"))
            + list(extract_dir.rglob("*.grib2"))
            + list(extract_dir.rglob("*.grb"))
        )

        if not candidates:
            raise ValueError(f"ZIP sin NetCDF/GRIB interno: {path}")

        return open_era5_any(candidates[0])

    # Caso NetCDF clásico / NetCDF4 / HDF5
    if magic.startswith(b"CDF") or magic.startswith(b"\x89HDF"):
        last_error = None

        for engine in ["netcdf4", "h5netcdf", "scipy"]:
            try:
                return xr.open_dataset(path, engine=engine)
            except Exception as e:
                last_error = e

        raise last_error

    # Caso GRIB
    if magic.startswith(b"GRIB"):
        import cfgrib

        datasets = cfgrib.open_datasets(
            str(path),
            backend_kwargs={
                "indexpath": "",
            },
        )

        if not datasets:
            raise ValueError(f"cfgrib no pudo abrir: {path}")

        cleaned = []

        for d in datasets:
            # Eliminar coordenadas escalares que suelen impedir merge:
            # heightAboveGround=10 para viento, heightAboveGround=2 para temperatura, etc.
            drop_coords = []

            for c in list(d.coords):
                if c not in d.dims and c not in [
                    "time",
                    "valid_time",
                    "step",
                    "latitude",
                    "longitude",
                    "lat",
                    "lon",
                ]:
                    drop_coords.append(c)

            if drop_coords:
                d = d.drop_vars(drop_coords, errors="ignore")

            cleaned.append(d)

        if len(cleaned) == 1:
            return cleaned[0]

        return xr.merge(cleaned, compat="override", join="outer")

    # Último intento genérico
    try:
        return xr.open_dataset(path)
    except Exception as e:
        raise ValueError(
            f"No se pudo abrir {path.name}. Magic bytes={magic}. Error original={repr(e)}"
        )


def find_coord_name(ds_in, candidates):
    for c in candidates:
        if c in ds_in.coords or c in ds_in.dims or c in ds_in.variables:
            return c
    return None


def find_var_name(ds_in, aliases):
    available = set(ds_in.data_vars) | set(ds_in.variables)

    for alias in aliases:
        if alias in available:
            return alias

    norm_map = {normalize_col(v): v for v in available}

    for alias in aliases:
        alias_norm = normalize_col(alias)
        if alias_norm in norm_map:
            return norm_map[alias_norm]

    return None


def rename_era5_vars(ds_in):
    rename_dict = {}

    for canonical, aliases in ERA5_VAR_ALIASES.items():
        found = find_var_name(ds_in, aliases)

        if found is not None and found != canonical:
            rename_dict[found] = canonical

    if rename_dict:
        ds_in = ds_in.rename(rename_dict)

    return ds_in


def standardize_era5_dataset(path):
    """
    Abre un archivo ERA5 y lo deja con:
    - coordenadas time, lat, lon
    - longitudes -180..180
    - recorte bbox Canarias
    - variables renombradas a u10, v10, sp, t2m, tp, swh, mwp, mwd si existen
    """

    path = Path(path)

    ds_in = open_era5_any(path)
    ds_in = rename_era5_vars(ds_in)

    # Resolver dimensiones extra comunes.
    for extra_dim in ["expver"]:
        if extra_dim in ds_in.dims:
            ds_in = ds_in.mean(dim=extra_dim, skipna=True)

    for extra_dim in ["number"]:
        if extra_dim in ds_in.dims:
            ds_in = ds_in.isel({extra_dim: 0})

    # Resolver tiempo.
    time_name = find_coord_name(ds_in, ["valid_time", "time", "forecast_time"])

    if time_name is None:
        raise ValueError(f"No se encontró coordenada temporal en {path.name}. Coords={list(ds_in.coords)}")

    # Si valid_time depende de time, lo usamos como coordenada temporal principal.
    if time_name == "valid_time" and "time" in ds_in.dims:
        if "time" in ds_in["valid_time"].dims:
            ds_in = ds_in.assign_coords(time=ds_in["valid_time"].values)
            ds_in = ds_in.drop_vars("valid_time", errors="ignore")

    elif time_name != "time":
        ds_in = ds_in.rename({time_name: "time"})

    lat_name = find_coord_name(ds_in, ["latitude", "lat"])
    lon_name = find_coord_name(ds_in, ["longitude", "lon"])

    if lat_name is None or lon_name is None:
        raise ValueError(
            f"No se encontraron coordenadas lat/lon en {path.name}. "
            f"Coords={list(ds_in.coords)} Dims={dict(ds_in.dims)}"
        )

    rename_coords = {}

    if lat_name != "lat":
        rename_coords[lat_name] = "lat"

    if lon_name != "lon":
        rename_coords[lon_name] = "lon"

    if rename_coords:
        ds_in = ds_in.rename(rename_coords)

    # Convertir longitudes 0..360 a -180..180 y ordenar.
    lon_std = standardize_longitudes_to_180(ds_in["lon"].values)
    ds_in = ds_in.assign_coords(lon=lon_std)
    ds_in = ds_in.sortby("lon")

    # Ordenar latitudes para hacer slice robusto.
    ds_in = ds_in.sortby("lat")

    ds_in = ds_in.sel(
        lat=slice(BBOX_CANARIAS["lat_min"], BBOX_CANARIAS["lat_max"]),
        lon=slice(BBOX_CANARIAS["lon_min"], BBOX_CANARIAS["lon_max"]),
    )

    if ds_in.sizes.get("lat", 0) == 0 or ds_in.sizes.get("lon", 0) == 0:
        raise ValueError(
            f"Recorte bbox vacío en {path.name}. "
            f"lat={ds_in.sizes.get('lat', 0)} lon={ds_in.sizes.get('lon', 0)}"
        )

    return ds_in


def dataset_to_flat_dataframe(ds_in, wanted_vars):
    """
    Convierte a DataFrame plano solo con las variables disponibles solicitadas.
    """

    available_vars = [v for v in wanted_vars if v in ds_in.data_vars]

    if not available_vars:
        print("Variables disponibles:", list(ds_in.data_vars))
        print("Variables buscadas:", wanted_vars)
        return pd.DataFrame()

    subset = ds_in[available_vars]

    df = subset.to_dataframe().reset_index()

    df = df.rename(columns={"time": "timestamp"})
    df["timestamp"] = ensure_utc(df["timestamp"])

    df = df.dropna(subset=["timestamp", "lat", "lon"]).copy()

    df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
    df["lon"] = pd.to_numeric(df["lon"], errors="coerce")

    df = df.dropna(subset=["lat", "lon"]).copy()

    return df


def build_grid_zone_map(df_grid):
    """
    Asigna cada punto de malla ERA5 a la playa/zona más cercana.
    """

    unique_points = (
        df_grid[["lat", "lon"]]
        .drop_duplicates()
        .reset_index(drop=True)
        .copy()
    )

    unique_points["era5_point_id"] = make_era5_point_id(
        unique_points["lat"],
        unique_points["lon"],
    )

    gdf_points = gpd.GeoDataFrame(
        unique_points,
        geometry=gpd.points_from_xy(unique_points["lon"], unique_points["lat"]),
        crs="EPSG:4326",
    )

    gdf_points_m = gdf_points.to_crs("EPSG:3857")

    nearest = gpd.sjoin_nearest(
        gdf_points_m,
        gdf_zones_m[["zona_id", "nombre_zona", "isla", "municipio", "geometry"]],
        how="left",
        distance_col="distance_to_zona_m",
    )

    nearest = (
        nearest
        .sort_values("distance_to_zona_m")
        .groupby("era5_point_id", as_index=False)
        .first()
    )

    point_zone = pd.DataFrame(nearest.drop(columns="geometry", errors="ignore"))
    point_zone["distance_to_zona_km"] = point_zone["distance_to_zona_m"] / 1000

    point_zone = point_zone[
        [
            "era5_point_id",
            "lat",
            "lon",
            "zona_id",
            "nombre_zona",
            "isla",
            "municipio",
            "distance_to_zona_km",
        ]
    ].copy()

    return point_zone

## Celda 8 — Flags de calidad y escritura incremental

In [ ]:
VARIABLE_RANGES = {
    # ocean
    "hs": (0, 15),
    "tp": (0, 35),
    "tm02": (0, 35),
    "wave_direction": (0, 360),
    "swell_height": (0, 15),
    "swell_period": (0, 35),
    "swell_direction": (0, 360),
    "wind_wave_height": (0, 15),
    "wind_wave_period": (0, 35),
    "stokes_drift": (0, 5),
    # meteo
    "wind_speed": (0, 60),
    "wind_direction": (0, 360),
    "u10": (-60, 60),
    "v10": (-60, 60),
    "pressure": (800, 1100),
    "temperature_air": (-10, 50),
    "precipitation": (0, 500),
    "humidity": (0, 100),
}


def add_quality_flags(df, variable_ranges):
    df = df.copy()

    for col, (vmin, vmax) in variable_ranges.items():
        if col not in df.columns:
            continue

        flag_col = f"{col}_flag"
        df[flag_col] = 0

        missing_mask = df[col].isna()
        outlier_mask = (~missing_mask) & ((df[col] < vmin) | (df[col] > vmax))

        df.loc[missing_mask, flag_col] = 1
        df.loc[outlier_mask, flag_col] = 2

        df[flag_col] = df[flag_col].astype("int8")

    return df


def remove_existing_source_partition(base_dir, source_name=SOURCE_NAME):
    source_path = base_dir / f"source={source_name}"
    if source_path.exists():
        shutil.rmtree(source_path)
        print("Eliminada partición antigua:", source_path)


def write_partitioned_parquet(df, base_dir, table_name):
    if df.empty:
        print(f"{table_name}: chunk vacío. No se guarda.")
        return

    df = df.copy()

    df["source"] = df["source"].fillna(SOURCE_NAME).astype(str)
    df["isla"] = df["isla"].fillna("ISLA_DESCONOCIDA").astype(str)
    df["year"] = df["year"].astype("int64")

    table = pa.Table.from_pandas(df, preserve_index=False)

    pq.write_to_dataset(
        table,
        root_path=str(base_dir),
        partition_cols=["source", "year", "isla"],
        compression="snappy",
    )


def validate_chunk_temporal(df, table_name, filename):
    if df.empty:
        return

    if df["timestamp"].isna().any():
        raise ValueError(f"{table_name} {filename}: timestamps nulos.")

    invalid = ~df["timestamp"].between(MIN_VALID_TS, MAX_VALID_TS)

    if invalid.any():
        raise ValueError(
            f"{table_name} {filename}: timestamps fuera de rango "
            f"{df.loc[invalid, 'timestamp'].min()} - {df.loc[invalid, 'timestamp'].max()}"
        )


def summarize_chunk(df, table_name, filename):
    rows = []

    base = {
        "table": table_name,
        "filename": filename,
        "rows": len(df),
        "timestamp_min": df["timestamp"].min() if len(df) and "timestamp" in df.columns else pd.NaT,
        "timestamp_max": df["timestamp"].max() if len(df) and "timestamp" in df.columns else pd.NaT,
        "unique_era5_points": df["era5_point_id"].nunique() if "era5_point_id" in df.columns else 0,
        "unique_zona_id": df["zona_id"].nunique() if "zona_id" in df.columns else 0,
        "missing_zona_id": int(df["zona_id"].isna().sum()) if "zona_id" in df.columns else np.nan,
    }

    rows.append(base)

    for col in df.columns:
        if col.endswith("_flag"):
            rows.append(
                {
                    "table": table_name,
                    "filename": filename,
                    "metric": f"{col}_missing_pct",
                    "value": float((df[col] == 1).mean() * 100) if len(df) else 0.0,
                }
            )
            rows.append(
                {
                    "table": table_name,
                    "filename": filename,
                    "metric": f"{col}_outlier_pct",
                    "value": float((df[col] == 2).mean() * 100) if len(df) else 0.0,
                }
            )

    return rows


def missing_pct(df, col):
    if df.empty or col not in df.columns:
        return 100.0
    return float(df[col].isna().mean() * 100)

## Celda 9 — Transformaciones ERA5 → `ocean_hourly` y `meteo_hourly`

In [ ]:
OCEAN_RAW_VARS = ["swh", "mwp", "mwd"]
METEO_RAW_VARS = ["u10", "v10", "sp", "t2m", "tp"]

OCEAN_COLUMNS = [
    "timestamp",
    "zona_id",
    "era5_point_id",
    "lat",
    "lon",
    "source",
    "hs",
    "hmax",
    "tp",
    "tm02",
    "wave_direction",
    "swell_height",
    "swell_period",
    "swell_direction",
    "wind_wave_height",
    "wind_wave_period",
    "stokes_drift",
    "distance_to_zona_km",
    "temporal_resolution",
    "year",
    "isla",
]

METEO_COLUMNS = [
    "timestamp",
    "station_id",
    "zona_id",
    "era5_point_id",
    "lat",
    "lon",
    "source",
    "wind_speed",
    "wind_direction",
    "wind_gust",
    "temperature_air",
    "pressure",
    "precipitation",
    "humidity",
    "u10",
    "v10",
    "distance_to_zona_km",
    "temporal_resolution",
    "year",
    "isla",
]


def make_ocean_hourly_chunk(df_raw, point_zone):
    if df_raw.empty or not any(v in df_raw.columns for v in OCEAN_RAW_VARS):
        return pd.DataFrame(columns=OCEAN_COLUMNS)

    df = df_raw[["timestamp", "lat", "lon"] + [v for v in OCEAN_RAW_VARS if v in df_raw.columns]].copy()

    df["era5_point_id"] = make_era5_point_id(df["lat"], df["lon"])

    df = df.merge(
        point_zone[
            [
                "era5_point_id",
                "zona_id",
                "nombre_zona",
                "isla",
                "municipio",
                "distance_to_zona_km",
            ]
        ],
        on="era5_point_id",
        how="left",
    )

    df["source"] = SOURCE_NAME
    df["temporal_resolution"] = "hourly"
    df["year"] = df["timestamp"].dt.year.astype("Int64")

    # ERA5 ocean mapping.
    df["hs"] = pd.to_numeric(df["swh"], errors="coerce") if "swh" in df.columns else np.nan
    df["tp"] = pd.to_numeric(df["mwp"], errors="coerce") if "mwp" in df.columns else np.nan
    df["wave_direction"] = pd.to_numeric(df["mwd"], errors="coerce") % 360 if "mwd" in df.columns else np.nan

    # Variables no disponibles en esta descarga ERA5.
    df["hmax"] = np.nan
    df["tm02"] = np.nan
    df["swell_height"] = np.nan
    df["swell_period"] = np.nan
    df["swell_direction"] = np.nan
    df["wind_wave_height"] = np.nan
    df["wind_wave_period"] = np.nan
    df["stokes_drift"] = np.nan

    for col in OCEAN_COLUMNS:
        if col not in df.columns:
            df[col] = np.nan

    df = df[OCEAN_COLUMNS].copy()

    df = (
        df
        .sort_values(["era5_point_id", "timestamp"])
        .drop_duplicates(subset=["era5_point_id", "timestamp", "source"], keep="first")
    )

    df = add_quality_flags(df, VARIABLE_RANGES)

    return df


def make_meteo_hourly_chunk(df_raw, point_zone):
    if df_raw.empty or not any(v in df_raw.columns for v in METEO_RAW_VARS):
        return pd.DataFrame(columns=METEO_COLUMNS)

    df = df_raw[["timestamp", "lat", "lon"] + [v for v in METEO_RAW_VARS if v in df_raw.columns]].copy()

    df["era5_point_id"] = make_era5_point_id(df["lat"], df["lon"])

    df = df.merge(
        point_zone[
            [
                "era5_point_id",
                "zona_id",
                "nombre_zona",
                "isla",
                "municipio",
                "distance_to_zona_km",
            ]
        ],
        on="era5_point_id",
        how="left",
    )

    df["source"] = SOURCE_NAME
    df["temporal_resolution"] = "hourly"
    df["station_id"] = df["era5_point_id"]
    df["year"] = df["timestamp"].dt.year.astype("Int64")

    if {"u10", "v10"}.issubset(df.columns):
        df["wind_speed"] = wind_speed_from_uv(df["u10"], df["v10"])
        df["wind_direction"] = wind_direction_from_uv(df["u10"], df["v10"])
    else:
        df["wind_speed"] = np.nan
        df["wind_direction"] = np.nan

    # Temperatura: Kelvin → Celsius si procede.
    if "t2m" in df.columns:
        temperature = pd.to_numeric(df["t2m"], errors="coerce")
        if temperature.median(skipna=True) > 100:
            temperature = temperature - 273.15
        df["temperature_air"] = temperature
    else:
        df["temperature_air"] = np.nan

    # Presión: Pa → hPa si procede.
    if "sp" in df.columns:
        pressure = pd.to_numeric(df["sp"], errors="coerce")
        if pressure.median(skipna=True) > 2000:
            pressure = pressure / 100.0
        df["pressure"] = pressure
    else:
        df["pressure"] = np.nan

    # Precipitación: ERA5 tp suele venir en metros. Silver: mm.
    if "tp" in df.columns:
        precipitation = pd.to_numeric(df["tp"], errors="coerce")
        if precipitation.quantile(0.99) < 10:
            precipitation = precipitation * 1000.0
        df["precipitation"] = precipitation
    else:
        df["precipitation"] = np.nan

    # No disponible en esta descarga.
    df["wind_gust"] = np.nan
    df["humidity"] = np.nan

    for col in METEO_COLUMNS:
        if col not in df.columns:
            df[col] = np.nan

    df = df[METEO_COLUMNS].copy()

    df = (
        df
        .sort_values(["era5_point_id", "timestamp"])
        .drop_duplicates(subset=["era5_point_id", "timestamp", "source"], keep="first")
    )

    df = add_quality_flags(df, VARIABLE_RANGES)

    return df

## Celda 10 — Borrar partición ERA5 antigua antes de procesar

In [ ]:
# Evita mezclar una ejecución antigua con la nueva.
remove_existing_source_partition(OUT_OCEAN_DIR, SOURCE_NAME)
remove_existing_source_partition(OUT_METEO_DIR, SOURCE_NAME)

print("Particiones ERA5 antiguas eliminadas si existían.")

Eliminada partición antigua: /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/meteo_hourly/source=ERA5
Particiones ERA5 antiguas eliminadas si existían.


## Celda 11 — Procesamiento incremental de todos los NetCDF ERA5

In [ ]:
file_summaries = []
quality_rows = []
grid_zone_maps = []
processing_errors = []

for p in tqdm(era5_files, desc="Procesando ERA5"):
    try:
        with standardize_era5_dataset(p) as ds_month:
            available_vars = list(ds_month.data_vars)

            all_wanted = sorted(set(OCEAN_RAW_VARS + METEO_RAW_VARS))
            df_raw = dataset_to_flat_dataframe(ds_month, all_wanted)

        if df_raw.empty:
            processing_errors.append(
                {
                    "filename": p.name,
                    "path": str(p),
                    "error": "No hay variables ERA5 esperadas en el archivo.",
                }
            )
            continue

        # Validación temporal temprana.
        invalid_ts = ~df_raw["timestamp"].between(MIN_VALID_TS, MAX_VALID_TS)
        if invalid_ts.any():
            raise ValueError(
                f"Timestamps fuera de rango en raw: "
                f"{df_raw.loc[invalid_ts, 'timestamp'].min()} - {df_raw.loc[invalid_ts, 'timestamp'].max()}"
            )

        point_zone = build_grid_zone_map(df_raw[["lat", "lon"]].drop_duplicates())

        grid_zone_maps.append(point_zone)

        ocean_chunk = make_ocean_hourly_chunk(df_raw, point_zone)
        meteo_chunk = make_meteo_hourly_chunk(df_raw, point_zone)

        validate_chunk_temporal(ocean_chunk, "ocean_hourly", p.name)
        validate_chunk_temporal(meteo_chunk, "meteo_hourly", p.name)

        if not ocean_chunk.empty:
            if ocean_chunk["zona_id"].isna().any():
                raise ValueError(f"{p.name}: ocean_chunk tiene zona_id nulos.")
            write_partitioned_parquet(ocean_chunk, OUT_OCEAN_DIR, "ocean_hourly")

        if not meteo_chunk.empty:
            if meteo_chunk["zona_id"].isna().any():
                raise ValueError(f"{p.name}: meteo_chunk tiene zona_id nulos.")
            write_partitioned_parquet(meteo_chunk, OUT_METEO_DIR, "meteo_hourly")

        file_summaries.append(
            {
                "filename": p.name,
                "path": str(p),
                "raw_rows": len(df_raw),
                "ocean_rows": len(ocean_chunk),
                "meteo_rows": len(meteo_chunk),
                "timestamp_min": df_raw["timestamp"].min(),
                "timestamp_max": df_raw["timestamp"].max(),
                "unique_timestamps": df_raw["timestamp"].nunique(),
                "unique_era5_points": point_zone["era5_point_id"].nunique(),
                "available_vars": json.dumps(available_vars, ensure_ascii=False),
                "ocean_key_hs_missing_pct": missing_pct(ocean_chunk, "hs") if not ocean_chunk.empty else np.nan,
                "ocean_key_tp_missing_pct": missing_pct(ocean_chunk, "tp") if not ocean_chunk.empty else np.nan,
                "meteo_key_wind_speed_missing_pct": missing_pct(meteo_chunk, "wind_speed") if not meteo_chunk.empty else np.nan,
                "meteo_key_temperature_air_missing_pct": missing_pct(meteo_chunk, "temperature_air") if not meteo_chunk.empty else np.nan,
                "meteo_key_pressure_missing_pct": missing_pct(meteo_chunk, "pressure") if not meteo_chunk.empty else np.nan,
            }
        )

        quality_rows.extend(summarize_chunk(ocean_chunk, "ocean_hourly", p.name))
        quality_rows.extend(summarize_chunk(meteo_chunk, "meteo_hourly", p.name))

        del df_raw, point_zone, ocean_chunk, meteo_chunk
        gc.collect()

    except Exception as e:
        processing_errors.append(
            {
                "filename": p.name,
                "path": str(p),
                "error": repr(e),
            }
        )

file_summary_df = pd.DataFrame(file_summaries)
processing_errors_df = pd.DataFrame(processing_errors)
quality_chunks_df = pd.DataFrame(quality_rows)

print("Archivos procesados correctamente:", len(file_summary_df))
print("Errores de procesamiento:", len(processing_errors_df))

display(file_summary_df.head())
display(processing_errors_df)

file_summary_df.to_csv(QC_DIR / "quality_era5_file_summary.csv", index=False)
processing_errors_df.to_csv(QC_DIR / "quality_era5_processing_errors.csv", index=False)
quality_chunks_df.to_csv(QC_DIR / "quality_era5_chunks_summary.csv", index=False)

if len(processing_errors_df):
    raise ValueError("Hay errores de procesamiento ERA5. Revisar quality_era5_processing_errors.csv.")

Procesando ERA5:   0%|          | 0/300 [00:00<?, ?it/s]

Archivos procesados correctamente: 300
Errores de procesamiento: 0


,filename,path,raw_rows,ocean_rows,meteo_rows,timestamp_min,timestamp_max,unique_timestamps,unique_era5_points,available_vars,ocean_key_hs_missing_pct,ocean_key_tp_missing_pct,meteo_key_wind_speed_missing_pct,meteo_key_temperature_air_missing_pct,meteo_key_pressure_missing_pct
0,era5_canarias_2001_01.nc,/content/drive/MyDrive/AI Projects/DeepWave Ca...,188232,0,188232,2001-01-01 00:00:00+00:00,2001-01-31 23:00:00+00:00,744,253,"[""tp""]",NaN,NaN,100.0,100.0,100.0
1,era5_canarias_2001_02.nc,/content/drive/MyDrive/AI Projects/DeepWave Ca...,170016,0,170016,2001-02-01 00:00:00+00:00,2001-02-28 23:00:00+00:00,672,253,"[""tp""]",NaN,NaN,100.0,100.0,100.0
2,era5_canarias_2001_03.nc,/content/drive/MyDrive/AI Projects/DeepWave Ca...,188232,0,188232,2001-03-01 00:00:00+00:00,2001-03-31 23:00:00+00:00,744,253,"[""tp""]",NaN,NaN,100.0,100.0,100.0
3,era5_canarias_2001_04.nc,/content/drive/MyDrive/AI Projects/DeepWave Ca...,182160,0,182160,2001-04-01 00:00:00+00:00,2001-04-30 23:00:00+00:00,720,253,"[""tp""]",NaN,NaN,100.0,100.0,100.0
4,era5_canarias_2001_05.nc,/content/drive/MyDrive/AI Projects/DeepWave Ca...,188232,0,188232,2001-05-01 00:00:00+00:00,2001-05-31 23:00:00+00:00,744,253,"[""tp""]",NaN,NaN,100.0,100.0,100.0


""


## Celda 12 — Guardar metadata de puntos ERA5 → zonas

In [ ]:
if grid_zone_maps:
    era5_point_to_zone = (
        pd.concat(grid_zone_maps, ignore_index=True)
        .sort_values(["era5_point_id", "distance_to_zona_km"])
        .drop_duplicates(subset=["era5_point_id"], keep="first")
        .reset_index(drop=True)
    )
else:
    era5_point_to_zone = pd.DataFrame()

print("Puntos ERA5 únicos:", len(era5_point_to_zone))

if len(era5_point_to_zone):
    display(era5_point_to_zone.head())
    print("Distancia ERA5 → zona más cercana, km:")
    display(era5_point_to_zone["distance_to_zona_km"].describe())

era5_point_to_zone.to_csv(META_DIR / "era5_point_to_zone.csv", index=False)

Puntos ERA5 únicos: 253


,era5_point_id,lat,lon,zona_id,nombre_zona,isla,municipio,distance_to_zona_km
0,ERA5_27.0_-13.0,27.0,-13.00,CAN_FV_LAS_PLAYITAS,Las Playitas,Fuerteventura,Tuineje,189.417610
1,ERA5_27.0_-13.25,27.0,-13.25,CAN_FV_GRAN_TARAJAL,Gran Tarajal,Fuerteventura,Tuineje,174.622169
2,ERA5_27.0_-13.5,27.0,-13.50,CAN_FV_MORRO_JABLE,Morro Jable,Fuerteventura,Pájara,160.521567
3,ERA5_27.0_-13.75,27.0,-13.75,CAN_FV_MORRO_JABLE,Morro Jable,Fuerteventura,Pájara,146.233077
4,ERA5_27.0_-14.0,27.0,-14.00,CAN_FV_MORRO_JABLE,Morro Jable,Fuerteventura,Pájara,136.198590


Distancia ERA5 → zona más cercana, km:


,distance_to_zona_km
count,253.000000
mean,61.568350
std,40.952486
min,1.062900
25%,27.211641
50%,55.873820
75%,92.560223
max,189.417610


## Celda 13 — Resumen global de calidad ERA5

In [ ]:
def dataset_count_and_sample(path, source_name=SOURCE_NAME, sample_n=5):
    if not path.exists():
        return 0, pd.DataFrame()

    dataset = ds.dataset(str(path), format="parquet", partitioning="hive")
    count = dataset.count_rows(filter=(ds.field("source") == source_name))

    if count == 0:
        return 0, pd.DataFrame()

    sample = dataset.head(sample_n, filter=(ds.field("source") == source_name)).to_pandas()

    return count, sample


ocean_count, ocean_sample = dataset_count_and_sample(OUT_OCEAN_DIR)
meteo_count, meteo_sample = dataset_count_and_sample(OUT_METEO_DIR)

print("Filas guardadas ocean_hourly ERA5:", ocean_count)
print("Filas guardadas meteo_hourly ERA5:", meteo_count)

if len(ocean_sample):
    print("Sample ocean_hourly:")
    display(ocean_sample)

if len(meteo_sample):
    print("Sample meteo_hourly:")
    display(meteo_sample)

global_quality = pd.DataFrame(
    [
        {
            "table": "ocean_hourly",
            "source": SOURCE_NAME,
            "rows": ocean_count,
            "files_processed": len(file_summary_df),
            "timestamp_min": file_summary_df["timestamp_min"].min() if len(file_summary_df) else pd.NaT,
            "timestamp_max": file_summary_df["timestamp_max"].max() if len(file_summary_df) else pd.NaT,
            "mean_hs_missing_pct_by_file": file_summary_df["ocean_key_hs_missing_pct"].mean() if "ocean_key_hs_missing_pct" in file_summary_df else np.nan,
            "mean_tp_missing_pct_by_file": file_summary_df["ocean_key_tp_missing_pct"].mean() if "ocean_key_tp_missing_pct" in file_summary_df else np.nan,
        },
        {
            "table": "meteo_hourly",
            "source": SOURCE_NAME,
            "rows": meteo_count,
            "files_processed": len(file_summary_df),
            "timestamp_min": file_summary_df["timestamp_min"].min() if len(file_summary_df) else pd.NaT,
            "timestamp_max": file_summary_df["timestamp_max"].max() if len(file_summary_df) else pd.NaT,
            "mean_wind_speed_missing_pct_by_file": file_summary_df["meteo_key_wind_speed_missing_pct"].mean() if "meteo_key_wind_speed_missing_pct" in file_summary_df else np.nan,
            "mean_temperature_air_missing_pct_by_file": file_summary_df["meteo_key_temperature_air_missing_pct"].mean() if "meteo_key_temperature_air_missing_pct" in file_summary_df else np.nan,
            "mean_pressure_missing_pct_by_file": file_summary_df["meteo_key_pressure_missing_pct"].mean() if "meteo_key_pressure_missing_pct" in file_summary_df else np.nan,
        },
    ]
)

display(global_quality)

global_quality.to_csv(QC_DIR / "quality_era5_global_summary.csv", index=False)

Filas guardadas ocean_hourly ERA5: 0
Filas guardadas meteo_hourly ERA5: 55443432
Sample meteo_hourly:


,timestamp,station_id,zona_id,era5_point_id,lat,lon,wind_speed,wind_direction,wind_gust,temperature_air,...,wind_direction_flag,u10_flag,v10_flag,pressure_flag,temperature_air_flag,precipitation_flag,humidity_flag,source,year,isla
0,2001-10-01 00:00:00+00:00,ERA5_29.5_-13.0,CAN_LZ_ISLA_DE_LA_ALEGRANZA,ERA5_29.5_-13.0,29.5,-13.0,NaN,NaN,NaN,NaN,...,1,1,1,1,1,0,1,ERA5,2001,Alegranza
1,2001-10-01 01:00:00+00:00,ERA5_29.5_-13.0,CAN_LZ_ISLA_DE_LA_ALEGRANZA,ERA5_29.5_-13.0,29.5,-13.0,NaN,NaN,NaN,NaN,...,1,1,1,1,1,0,1,ERA5,2001,Alegranza
2,2001-10-01 02:00:00+00:00,ERA5_29.5_-13.0,CAN_LZ_ISLA_DE_LA_ALEGRANZA,ERA5_29.5_-13.0,29.5,-13.0,NaN,NaN,NaN,NaN,...,1,1,1,1,1,0,1,ERA5,2001,Alegranza
3,2001-10-01 03:00:00+00:00,ERA5_29.5_-13.0,CAN_LZ_ISLA_DE_LA_ALEGRANZA,ERA5_29.5_-13.0,29.5,-13.0,NaN,NaN,NaN,NaN,...,1,1,1,1,1,0,1,ERA5,2001,Alegranza
4,2001-10-01 04:00:00+00:00,ERA5_29.5_-13.0,CAN_LZ_ISLA_DE_LA_ALEGRANZA,ERA5_29.5_-13.0,29.5,-13.0,NaN,NaN,NaN,NaN,...,1,1,1,1,1,0,1,ERA5,2001,Alegranza


,table,source,rows,files_processed,timestamp_min,timestamp_max,mean_hs_missing_pct_by_file,mean_tp_missing_pct_by_file,mean_wind_speed_missing_pct_by_file,mean_temperature_air_missing_pct_by_file,mean_pressure_missing_pct_by_file
0,ocean_hourly,ERA5,0,300,2001-01-01 00:00:00+00:00,2025-12-31 23:00:00+00:00,NaN,NaN,NaN,NaN,NaN
1,meteo_hourly,ERA5,55443432,300,2001-01-01 00:00:00+00:00,2025-12-31 23:00:00+00:00,NaN,NaN,100.0,100.0,100.0


## Celda 14 — Validaciones finales antes de cerrar ERA5

In [ ]:
era5_available_vars = set()

for x in file_summary_df["available_vars"].dropna():
    try:
        vars_file = json.loads(x)
        era5_available_vars.update(vars_file)
    except Exception:
        pass

era5_available_vars = sorted(era5_available_vars)

print("Variables ERA5 disponibles:")
print(era5_available_vars)

has_precip = "tp" in era5_available_vars
has_meteo_core = all(v in era5_available_vars for v in ["u10", "v10", "t2m", "sp"])
has_ocean_core = all(v in era5_available_vars for v in ["swh", "mwp", "mwd"])

if len(file_summary_df) == 0:
    raise ValueError("No se procesó ningún archivo ERA5.")

if meteo_count == 0:
    raise ValueError("No se guardó ningún registro en meteo_hourly/source=ERA5.")

ts_min = pd.to_datetime(file_summary_df["timestamp_min"], utc=True, errors="coerce").min()
ts_max = pd.to_datetime(file_summary_df["timestamp_max"], utc=True, errors="coerce").max()

print("Rango temporal ERA5:", ts_min, "→", ts_max)

if ts_min < MIN_VALID_TS or ts_max > MAX_VALID_TS:
    raise ValueError("Rango temporal ERA5 fuera de límites válidos.")

if not has_meteo_core:
    print(
        "AVISO: ERA5 actual no contiene el núcleo meteorológico completo "
        "(u10, v10, t2m, sp). Se conserva lo disponible y se generará ERA5_PROXY."
    )

if not has_ocean_core:
    print(
        "AVISO: ERA5 actual no contiene variables de oleaje "
        "(swh, mwp, mwd). No se genera ocean_hourly/source=ERA5."
    )

if era5_available_vars == ["tp"]:
    era5_status = "precipitation_only"
    era5_reason = "Los archivos ERA5 actuales solo contienen total_precipitation/tp."
else:
    era5_status = "partial"
    era5_reason = f"Variables disponibles: {era5_available_vars}"

era5_output_status = pd.DataFrame(
    [
        {
            "source": "ERA5",
            "table": "meteo_hourly",
            "status": era5_status,
            "reason": era5_reason,
            "rows": meteo_count,
            "available_vars": ",".join(era5_available_vars),
        },
        {
            "source": "ERA5",
            "table": "ocean_hourly",
            "status": "not_generated" if ocean_count == 0 else "generated",
            "reason": (
                "ERA5 files do not contain swh/mwp/mwd wave variables"
                if ocean_count == 0
                else "ok"
            ),
            "rows": ocean_count,
            "available_vars": ",".join(era5_available_vars),
        },
    ]
)

display(era5_output_status)

era5_output_status.to_csv(
    QC_DIR / "quality_era5_output_status.csv",
    index=False,
)

print("Validación final ERA5 superada con estado:", era5_status)

Variables ERA5 disponibles:
['tp']
Rango temporal ERA5: 2001-01-01 00:00:00+00:00 → 2025-12-31 23:00:00+00:00
AVISO: ERA5 actual no contiene el núcleo meteorológico completo (u10, v10, t2m, sp). Se conserva lo disponible y se generará ERA5_PROXY.
AVISO: ERA5 actual no contiene variables de oleaje (swh, mwp, mwd). No se genera ocean_hourly/source=ERA5.


,source,table,status,reason,rows,available_vars
0,ERA5,meteo_hourly,precipitation_only,Los archivos ERA5 actuales solo contienen tota...,55443432,tp
1,ERA5,ocean_hourly,not_generated,ERA5 files do not contain swh/mwp/mwd wave var...,0,tp


Validación final ERA5 superada con estado: precipitation_only


## Celda 15 — Configuración de `ERA5_PROXY`

Como la descarga ERA5 disponible solo contiene `tp`/precipitación, esta sección crea una fuente auxiliar **sintética y claramente etiquetada**:

```text
source = ERA5_PROXY
```

Las variables proxy quedan marcadas con flag `3 = proxy/sintético/interpolado`. La precipitación se conserva desde ERA5 real.

In [ ]:
PROXY_SOURCE_NAME = "ERA5_PROXY"

print("Creando proxy meteorológico sintético a partir de ERA5 precipitation-only.")
print("Source:", PROXY_SOURCE_NAME)

# Variables sintéticas que se crearán de forma determinista.
PROXY_SYNTHETIC_VARS = [
    "wind_speed",
    "wind_direction",
    "wind_gust",
    "temperature_air",
    "pressure",
    "humidity",
    "u10",
    "v10",
]

PROXY_REAL_VARS = ["precipitation"]

# Tamaño de lote para no cargar los 55M registros en memoria.
PROXY_BATCH_SIZE = 250_000

Creando proxy meteorológico sintético a partir de ERA5 precipitation-only.
Source: ERA5_PROXY


## Celda 16 — Funciones para generar meteorología proxy por lotes

In [ ]:
def island_temperature_base(isla):
    isla_norm = normalize_text(isla)

    base = {
        "TENERIFE": 21.0,
        "GRAN CANARIA": 21.5,
        "LANZAROTE": 22.0,
        "FUERTEVENTURA": 22.0,
        "LA PALMA": 20.5,
        "LA GOMERA": 21.0,
        "EL HIERRO": 20.5,
        "LA GRACIOSA": 22.0,
        "ALEGRANZA": 22.0,
    }

    return base.get(isla_norm, 21.0)


def generate_meteo_proxy_chunk(df):
    """
    Genera variables meteorológicas proxy de forma determinista:
    - ciclos horario y estacional
    - variación espacial suave por lat/lon
    - precipitación real conservada desde ERA5 tp
    """

    df = df.copy()

    # Asegurar columnas de partición si vienen como diccionario/categoría.
    if "source" in df.columns:
        df["source"] = df["source"].astype(str)

    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")

    if df["timestamp"].isna().any():
        raise ValueError("Hay timestamps nulos en un lote ERA5 antes de crear proxy.")

    ts = df["timestamp"]
    hour = ts.dt.hour.astype(float)
    dayofyear = ts.dt.dayofyear.astype(float)

    lat = pd.to_numeric(df["lat"], errors="coerce")
    lon = pd.to_numeric(df["lon"], errors="coerce")

    if lat.isna().any() or lon.isna().any():
        raise ValueError("Hay coordenadas nulas en un lote ERA5 antes de crear proxy.")

    island_base_temp = df["isla"].apply(island_temperature_base).astype(float)

    # Temperatura: base insular + ciclo estacional + ciclo diario + variación espacial.
    seasonal_temp = 3.0 * np.sin(2 * np.pi * (dayofyear - 172) / 365.25)
    daily_temp = 2.0 * np.sin(2 * np.pi * (hour - 8) / 24)
    spatial_temp = (np.sin(lat * 10) + np.cos(lon * 10)) * 0.4

    df["temperature_air"] = island_base_temp + seasonal_temp + daily_temp + spatial_temp

    # Viento alisio dominante NE: más fuerte en verano y durante horas diurnas.
    seasonal_wind = 1.5 + 1.2 * np.sin(2 * np.pi * (dayofyear - 150) / 365.25)
    daily_wind = 0.8 * np.sin(2 * np.pi * (hour - 11) / 24)
    wind_spatial = np.abs(np.sin(lon * 3)) + np.abs(np.cos(lat * 3))

    df["wind_speed"] = 5.5 + seasonal_wind + daily_wind + wind_spatial
    df["wind_speed"] = df["wind_speed"].clip(lower=0.2, upper=18.0)

    # Dirección meteorológica: desde NE aproximado, con variación suave.
    df["wind_direction"] = (
        45
        + 20 * np.sin(2 * np.pi * dayofyear / 365.25)
        + 10 * np.sin(2 * np.pi * hour / 24)
        + lat * 0.5
    ) % 360

    # Componentes u/v usando convención meteorológica: dirección desde donde sopla.
    wd_rad = np.deg2rad(df["wind_direction"])
    df["u10"] = -df["wind_speed"] * np.sin(wd_rad)
    df["v10"] = -df["wind_speed"] * np.cos(wd_rad)

    # Presión suave.
    df["pressure"] = (
        1015.0
        + 3.0 * np.cos(2 * np.pi * dayofyear / 365.25)
        + 1.5 * np.sin(2 * np.pi * hour / 24)
        + 0.2 * np.sin(lon)
    )

    # Humedad aproximada: más alta si hay precipitación.
    precipitation = pd.to_numeric(df["precipitation"], errors="coerce") if "precipitation" in df.columns else pd.Series(0, index=df.index)
    rain_effect = np.where(precipitation.fillna(0) > 0.1, 12, 0)

    df["humidity"] = (
        68
        + 10 * np.cos(2 * np.pi * hour / 24)
        + rain_effect
        + 4 * np.sin(2 * np.pi * dayofyear / 365.25)
    ).clip(35, 100)

    df["wind_gust"] = (df["wind_speed"] * 1.45).clip(lower=0.2, upper=30.0)

    df["source"] = PROXY_SOURCE_NAME
    df["temporal_resolution"] = "hourly"
    df["year"] = df["timestamp"].dt.year.astype("int64")
    df["isla"] = df["isla"].fillna("ISLA_DESCONOCIDA").astype(str)

    # Flags:
    # 0 = ok real
    # 1 = missing
    # 2 = outlier
    # 3 = proxy/sintético/interpolado
    for col in PROXY_SYNTHETIC_VARS:
        df[f"{col}_flag"] = 3

    for col in PROXY_REAL_VARS:
        flag_col = f"{col}_flag"
        df[flag_col] = 0
        if col in df.columns:
            df.loc[df[col].isna(), flag_col] = 1
        else:
            df[col] = np.nan
            df[flag_col] = 1

    # Validación física básica.
    df.loc[(df["wind_speed"] < 0) | (df["wind_speed"] > 60), "wind_speed_flag"] = 2
    df.loc[(df["wind_direction"] < 0) | (df["wind_direction"] > 360), "wind_direction_flag"] = 2
    df.loc[(df["temperature_air"] < -10) | (df["temperature_air"] > 50), "temperature_air_flag"] = 2
    df.loc[(df["pressure"] < 800) | (df["pressure"] > 1100), "pressure_flag"] = 2
    df.loc[(df["humidity"] < 0) | (df["humidity"] > 100), "humidity_flag"] = 2

    # Mantener el esquema de meteo_hourly.
    for col in METEO_COLUMNS:
        if col not in df.columns:
            df[col] = np.nan

    extra_flag_cols = [
        f"{c}_flag"
        for c in [
            "wind_speed",
            "wind_direction",
            "wind_gust",
            "temperature_air",
            "pressure",
            "precipitation",
            "humidity",
            "u10",
            "v10",
        ]
    ]

    keep_cols = METEO_COLUMNS + [c for c in extra_flag_cols if c in df.columns]

    return df[keep_cols].copy()


def remove_existing_proxy_partition():
    proxy_path = OUT_METEO_DIR / f"source={PROXY_SOURCE_NAME}"
    if proxy_path.exists():
        shutil.rmtree(proxy_path)
        print("Eliminada partición antigua:", proxy_path)


def write_proxy_chunk(df):
    if df.empty:
        return

    table = pa.Table.from_pandas(df, preserve_index=False)

    pq.write_to_dataset(
        table,
        root_path=str(OUT_METEO_DIR),
        partition_cols=["source", "year", "isla"],
        compression="snappy",
    )

## Celda 17 — Generar y guardar `silver/meteo_hourly/source=ERA5_PROXY/`

In [ ]:
remove_existing_proxy_partition()

era5_dataset = ds.dataset(
    str(OUT_METEO_DIR),
    format="parquet",
    partitioning="hive",
)

proxy_rows_written = 0
proxy_batches = 0
proxy_quality_rows = []

scanner = era5_dataset.scanner(
    filter=(ds.field("source") == SOURCE_NAME),
    batch_size=PROXY_BATCH_SIZE,
)

for batch in tqdm(scanner.to_batches(), desc="Generando ERA5_PROXY por lotes"):
    batch_df = batch.to_pandas()

    if batch_df.empty:
        continue

    # Asegurar que solo se usa ERA5 real.
    if "source" in batch_df.columns:
        batch_df = batch_df[batch_df["source"].astype(str) == SOURCE_NAME].copy()

    if batch_df.empty:
        continue

    proxy_chunk = generate_meteo_proxy_chunk(batch_df)

    # Validación temporal del lote.
    if proxy_chunk["timestamp"].isna().any():
        raise ValueError("ERA5_PROXY tiene timestamps nulos en un lote.")

    invalid_ts = ~proxy_chunk["timestamp"].between(MIN_VALID_TS, MAX_VALID_TS)

    if invalid_ts.any():
        raise ValueError(
            "ERA5_PROXY tiene timestamps fuera de rango: "
            f"{proxy_chunk.loc[invalid_ts, 'timestamp'].min()} - "
            f"{proxy_chunk.loc[invalid_ts, 'timestamp'].max()}"
        )

    write_proxy_chunk(proxy_chunk)

    proxy_rows_written += len(proxy_chunk)
    proxy_batches += 1

    proxy_quality_rows.append(
        {
            "batch": proxy_batches,
            "rows": len(proxy_chunk),
            "timestamp_min": proxy_chunk["timestamp"].min(),
            "timestamp_max": proxy_chunk["timestamp"].max(),
            "unique_zona_id": proxy_chunk["zona_id"].nunique(),
            "wind_speed_missing_pct": float(proxy_chunk["wind_speed"].isna().mean() * 100),
            "temperature_air_missing_pct": float(proxy_chunk["temperature_air"].isna().mean() * 100),
            "pressure_missing_pct": float(proxy_chunk["pressure"].isna().mean() * 100),
            "precipitation_missing_pct": float(proxy_chunk["precipitation"].isna().mean() * 100),
        }
    )

    del batch_df, proxy_chunk
    gc.collect()

print("Lotes ERA5_PROXY generados:", proxy_batches)
print("Filas ERA5_PROXY guardadas:", proxy_rows_written)

if proxy_rows_written == 0:
    raise ValueError("No se generó ningún registro ERA5_PROXY.")

proxy_batches_quality = pd.DataFrame(proxy_quality_rows)
display(proxy_batches_quality.head())

proxy_batches_quality.to_csv(
    QC_DIR / "quality_era5_proxy_batches.csv",
    index=False,
)

Generando ERA5_PROXY por lotes: 0it [00:00, ?it/s]

Lotes ERA5_PROXY generados: 9387
Filas ERA5_PROXY guardadas: 55443432


,batch,rows,timestamp_min,timestamp_max,unique_zona_id,wind_speed_missing_pct,temperature_air_missing_pct,pressure_missing_pct,precipitation_missing_pct
0,1,3720,2001-12-01 00:00:00+00:00,2001-12-31 23:00:00+00:00,1,0.0,0.0,0.0,0.0
1,2,3720,2001-07-01 00:00:00+00:00,2001-07-31 23:00:00+00:00,1,0.0,0.0,0.0,0.0
2,3,3720,2001-08-01 00:00:00+00:00,2001-08-31 23:00:00+00:00,1,0.0,0.0,0.0,0.0
3,4,3600,2001-06-01 00:00:00+00:00,2001-06-30 23:00:00+00:00,1,0.0,0.0,0.0,0.0
4,5,3360,2001-02-01 00:00:00+00:00,2001-02-28 23:00:00+00:00,1,0.0,0.0,0.0,0.0


## Celda 18 — Validación y reporte de calidad de `ERA5_PROXY`

In [ ]:
proxy_count, proxy_sample = dataset_count_and_sample(
    OUT_METEO_DIR,
    source_name=PROXY_SOURCE_NAME,
    sample_n=5,
)

print("Filas guardadas meteo_hourly ERA5_PROXY:", proxy_count)

if proxy_count == 0:
    raise ValueError("No se pudo leer ERA5_PROXY desde el dataset Parquet.")

display(proxy_sample)

proxy_quality = pd.DataFrame(
    [
        {
            "source": PROXY_SOURCE_NAME,
            "table": "meteo_hourly",
            "rows": proxy_count,
            "timestamp_min": proxy_batches_quality["timestamp_min"].min(),
            "timestamp_max": proxy_batches_quality["timestamp_max"].max(),
            "available_real_vars": "precipitation",
            "synthetic_vars": ",".join(PROXY_SYNTHETIC_VARS),
            "note": (
                "Proxy meteorológico sintético generado porque Bronze ERA5 solo contiene tp. "
                "Las variables proxy están marcadas con flag=3 y no deben usarse como observación real "
                "ni como referencia de validación."
            ),
        }
    ]
)

display(proxy_quality)

proxy_quality.to_csv(
    QC_DIR / "quality_era5_proxy_summary.csv",
    index=False,
)

proxy_missing_summary = pd.DataFrame(
    [
        {
            "variable": "wind_speed",
            "missing_pct_mean_by_batch": proxy_batches_quality["wind_speed_missing_pct"].mean(),
        },
        {
            "variable": "temperature_air",
            "missing_pct_mean_by_batch": proxy_batches_quality["temperature_air_missing_pct"].mean(),
        },
        {
            "variable": "pressure",
            "missing_pct_mean_by_batch": proxy_batches_quality["pressure_missing_pct"].mean(),
        },
        {
            "variable": "precipitation",
            "missing_pct_mean_by_batch": proxy_batches_quality["precipitation_missing_pct"].mean(),
        },
    ]
)

display(proxy_missing_summary)

proxy_missing_summary.to_csv(
    QC_DIR / "quality_era5_proxy_missing_by_variable.csv",
    index=False,
)

if proxy_missing_summary["missing_pct_mean_by_batch"].max() > 5:
    raise ValueError("ERA5_PROXY tiene demasiados nulos en variables clave.")

print("Validación ERA5_PROXY superada.")

Filas guardadas meteo_hourly ERA5_PROXY: 55443432


,timestamp,station_id,zona_id,era5_point_id,lat,lon,wind_speed,wind_direction,wind_gust,temperature_air,...,wind_direction_flag,u10_flag,v10_flag,pressure_flag,temperature_air_flag,precipitation_flag,humidity_flag,source,year,isla
0,2001-07-01 00:00:00+00:00,ERA5_29.5_-13.0,CAN_LZ_ISLA_DE_LA_ALEGRANZA,ERA5_29.5_-13.0,29.5,-13.0,9.244515,59.965026,13.404547,20.512651,...,3,3,3,3,3,0,3,ERA5_PROXY,2001,Alegranza
1,2001-07-01 01:00:00+00:00,ERA5_29.5_-13.0,CAN_LZ_ISLA_DE_LA_ALEGRANZA,ERA5_29.5_-13.0,29.5,-13.0,9.051570,62.553217,13.124777,20.312850,...,3,3,3,3,3,0,3,ERA5_PROXY,2001,Alegranza
2,2001-07-01 02:00:00+00:00,ERA5_29.5_-13.0,CAN_LZ_ISLA_DE_LA_ALEGRANZA,ERA5_29.5_-13.0,29.5,-13.0,8.885885,64.965026,12.884533,20.244702,...,3,3,3,3,3,0,3,ERA5_PROXY,2001,Alegranza
3,2001-07-01 03:00:00+00:00,ERA5_29.5_-13.0,CAN_LZ_ISLA_DE_LA_ALEGRANZA,ERA5_29.5_-13.0,29.5,-13.0,8.758750,67.036094,12.700187,20.312850,...,3,3,3,3,3,0,3,ERA5_PROXY,2001,Alegranza
4,2001-07-01 04:00:00+00:00,ERA5_29.5_-13.0,CAN_LZ_ISLA_DE_LA_ALEGRANZA,ERA5_29.5_-13.0,29.5,-13.0,8.678830,68.625280,12.584303,20.512651,...,3,3,3,3,3,0,3,ERA5_PROXY,2001,Alegranza


,source,table,rows,timestamp_min,timestamp_max,available_real_vars,synthetic_vars,note
0,ERA5_PROXY,meteo_hourly,55443432,2001-01-01 00:00:00+00:00,2025-12-31 23:00:00+00:00,precipitation,"wind_speed,wind_direction,wind_gust,temperatur...",Proxy meteorológico sintético generado porque ...


,variable,missing_pct_mean_by_batch
0,wind_speed,0.0
1,temperature_air,0.0
2,pressure,0.0
3,precipitation,0.0


Validación ERA5_PROXY superada.


## Celda 19 — Listado de salidas generadas

In [ ]:
print("Parquet ERA5 generado en:")
print("-", OUT_OCEAN_DIR / "source=ERA5", "(puede no existir si no hay oleaje en Bronze ERA5)")
print("-", OUT_METEO_DIR / "source=ERA5")
print("-", OUT_METEO_DIR / "source=ERA5_PROXY")

print("\nReportes de calidad ERA5:")
for p in sorted(QC_DIR.glob("quality_era5*.csv")):
    print("-", p)

print("\nMetadatos ERA5:")
for p in sorted(META_DIR.glob("era5*.csv")):
    print("-", p)

print("\nEstado final recomendado:")
print("- ERA5 real: precipitation-only si available_vars == ['tp']")
print("- ERA5_PROXY: meteorología sintética marcada con flags=3 para completar pipeline")

Parquet ERA5 generado en:
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/ocean_hourly/source=ERA5 (puede no existir si no hay oleaje en Bronze ERA5)
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/meteo_hourly/source=ERA5
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/meteo_hourly/source=ERA5_PROXY

Reportes de calidad ERA5:
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_era5_chunks_summary.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_era5_file_summary.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_era5_global_summary.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_era5_output_status.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_era5_processing_errors.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_rep

## Resultado esperado

Al terminar deberían existir:

```text
silver/meteo_hourly/source=ERA5/year=YYYY/isla=.../*.parquet
silver/meteo_hourly/source=ERA5_PROXY/year=YYYY/isla=.../*.parquet
silver/_quality_reports/quality_era5_file_summary.csv
silver/_quality_reports/quality_era5_global_summary.csv
silver/_quality_reports/quality_era5_output_status.csv
silver/_quality_reports/quality_era5_proxy_summary.csv
silver/_quality_reports/quality_era5_proxy_missing_by_variable.csv
silver/_metadata/era5_point_to_zone.csv
```

En este proyecto, la descarga ERA5 disponible contiene únicamente `tp`, por lo que:

```text
ERA5 real       → precipitación horaria
ERA5_PROXY      → viento, temperatura, presión, humedad y componentes u/v sintéticos
ocean_hourly ERA5 → no generado si no existen swh/mwp/mwd
```

Las variables sintéticas se conservan con:

```text
source = ERA5_PROXY
*_flag = 3
```

y deben documentarse como proxy para completar el flujo end-to-end, no como observaciones reales.